In [ ]:
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# 1. 수집할 키워드들
keywords = []
cnn_news_urls = set()

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

for keyword in keywords:
    print(f"🔗 [{keyword}] URL 수집 시작...")
    driver.get(f"https://www.cnn.com/search?q={keyword.replace(' ', '+')}&sort=newest&types=all")
    time.sleep(5)

    no_growth = 0
    max_no_growth = 5 # 5번 연속으로 새 링크가 안 나오면 다음 키워드로

    for i in range(30): # 최대 30번 스크롤/페이지 넘기기
        before = len(cnn_news_urls)
        
        # 화면에 보이는 모든 뉴스 링크(a 태그) 찾기
        # CNN 뉴스는 주소에 날짜(202x/xx/xx)가 포함된 것이 특징입니다.
        anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/202"]') 
        
        for a in anchors:
            href = a.get_attribute("href")
            if href and "cnn.com" in href:
                # 불필요한 파라미터(? 등) 제거하고 순수 URL만 저장
                cnn_news_urls.add(href.split("?")[0])

        after = len(cnn_news_urls)
        print(f"[Keyword: {keyword}] unique_urls={after}")

        if after == before:
            no_growth += 1
        else:
            no_growth = 0

        if no_growth >= max_no_growth:
            break

        # 다음 페이지 클릭 
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, ".pagination-arrow-right")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(3)
        except:
            break

# 리스트로 변환 및 중간 저장
cnn_news_urls = list(cnn_news_urls)
with open('cnn_news_urls.pkl', 'wb') as f:
    pickle.dump(cnn_news_urls, f)

print(f"✨ 총 {len(cnn_news_urls)}개의 뉴스 링크를 찾았습니다. 'cnn_news_urls.pkl' 저장 완료!")
driver.quit()

In [ ]:
import pandas as pd
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

# 1. 아까 저장한 파일 불러오기
with open('cnn_news_urls.pkl', 'rb') as f:
    cnn_news_urls = pickle.load(f)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
results = []

start_index = 0
end_index = len(cnn_news_urls) 
batch_urls = cnn_news_urls[start_index:end_index]

print(f"🚀 총 {len(batch_urls)}개의 모든 URL 수집을 시작합니다!")

for url in batch_urls:
    try:
        driver.get(url)
        time.sleep(2) # 상세 페이지 로딩 대기

        # CNN 상세 페이지의 제목과 날짜 선택자
        # 제목: .main_headline 또는 h1.headline__text
        # 날짜: .timestamp 또는 div.timestamp
        title = driver.find_element(By.CSS_SELECTOR, "h1.headline__text, .main_headline").text
        
        # 날짜는 CNN 특성상 여러 곳에 있을 수 있어 여러 시도
        try:
            date = driver.find_element(By.CSS_SELECTOR, ".timestamp, div.timestamp").text
        except:
            date = "Unknown Date"

        results.append({"제목": title, "날짜": date, "URL": url})
        print(f"✅ 수집 완료: {title[:30]}...")

    except:
        print(f"❌ 오류 발생 주소 (기사가 아니거나 로딩 실패): {url}")
        continue

# 최종 결과 저장
df = pd.DataFrame(results)
filename = f"CNN_Final_Data_{start_index}_{end_index}.csv"
df.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"\n✨ {len(results)}건 수집 완료! '{filename}' 파일을 확인하세요.")
driver.quit()